In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import psycopg2
from sqlalchemy import create_engine

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
username = "postgres"
password = "password"
host = "localhost"
port = "5432"
database = "anomaly_detection_db"

engine = create_engine(
    f"postgresql://{username}:{password}@{host}:{port}/{database}"
)

In [ ]:
query = "SELECT * FROM creditcard_fraud LIMIT 5;"

df_credit = pd.read_sql(query, engine)

df_credit.head()

In [ ]:
query = "SELECT * FROM creditcard_fraud;"

df_credit = pd.read_sql(query, engine)

df_credit.shape

In [ ]:
df_credit.info()

In [ ]:
df_credit.describe().T

In [ ]:
df_credit["Class"].value_counts()

In [ ]:
df_credit["Class"].value_counts(normalize=True) * 100

## Class Distribution Analysis

The dataset is highly imbalanced, with only 0.1727% fraudulent transactions and 99.8273% normal transactions.

This extreme imbalance makes anomaly detection approaches highly suitable for the fraud detection problem, since fraudulent transactions represent rare abnormal behavior compared to regular financial activity.

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x="Class", data=df_credit)
plt.title("Class Distribution")

plt.savefig("class_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
fraud_rate = (
    df_credit["Class"].value_counts(normalize=True)[1] * 100
)

print(f"Fraud Rate: %{fraud_rate:.4f}")

In [ ]:
df_credit.groupby("Class")["Amount"].describe()

## Transaction Amount Analysis

Fraudulent transactions tend to have higher average transaction amounts compared to normal transactions.

Although there is significant overlap between normal and fraudulent transactions, transaction amount may still provide useful anomaly signals for fraud detection models.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="Class", y="Amount", data=df_credit)
plt.title("Transaction Amount by Class")

plt.savefig("transaction_amount_boxplot.png", bbox_inches="tight")
plt.show()

## Log-Transformed Amount Analysis

Since transaction amounts are highly skewed, log transformation helps reveal the underlying distribution more clearly.

This improves interpretability and helps identify anomaly patterns hidden behind extreme outliers.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(
    x="Class",
    y=np.log1p(df_credit["Amount"]),
    data=df_credit
)

plt.title("Log-Transformed Amount by Class")

plt.savefig("log_amount_boxplot.png", bbox_inches="tight")
plt.show()

In [ ]:
df_credit.groupby("Class")["Amount"].mean()

In [ ]:
df_credit.groupby("Class")["Time"].describe()

## Transaction Time Analysis

Fraudulent transactions do not occur in a clearly isolated time window.

Although fraudulent transactions tend to appear slightly earlier on average, transaction time alone is not a strong standalone fraud indicator.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x="Class", y="Time", data=df_credit)
plt.title("Transaction Time by Class")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(
    data=df_credit,
    x="Time",
    hue="Class",
    bins=50,
    kde=True
)

plt.title("Time Distribution by Class")

plt.savefig("time_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
corr = df_credit.corr()

corr["Class"].sort_values(ascending=False)

In [ ]:
corr["Class"].sort_values(ascending=False).head(10)

## Correlation Analysis

Features V17, V14, and V12 show the strongest negative correlation with fraudulent transactions, indicating strong anomaly signals.

These variables are likely to play a significant role in anomaly detection models and fraud classification performance.

In [ ]:
top_features = corr["Class"].abs().sort_values(
    ascending=False
).head(10).index

plt.figure(figsize=(10,6))
sns.heatmap(
    df_credit[top_features].corr(),
    annot=True,
    cmap="coolwarm"
)

plt.title("Top Correlated Features with Fraud")

plt.savefig("correlation_heatmap.png", bbox_inches="tight")
plt.show()

## Data Quality Check

No missing values were found in the dataset.

However, 1,081 duplicate transactions (0.38%) were identified. These duplicate records may negatively affect anomaly detection performance and are considered for removal during preprocessing.

In [ ]:
df_credit.isnull().sum()

In [ ]:
df_credit.duplicated().sum()

In [ ]:
duplicate_rate = (
    df_credit.duplicated().sum() / len(df_credit)
) * 100

print(f"Duplicate Rate: %{duplicate_rate:.4f}")

## Duplicate Removal

No missing values were found in the dataset.

However, 1,081 duplicate transactions were identified and removed to reduce potential model bias and improve anomaly detection performance.

Duplicate transactions may cause the model to memorize repeated behavior instead of learning true anomaly patterns.

In [ ]:
print("Before duplicate removal:", df_credit.shape)

df_credit = df_credit.drop_duplicates()

print("After duplicate removal:", df_credit.shape)

In [ ]:
df_credit.duplicated().sum()

In [ ]:
df_credit["Class"].value_counts(normalize=True) * 100

## Feature Scaling

The features V1–V28 are already transformed using PCA and are approximately standardized.

However, Time and Amount remain on different numerical scales and may negatively affect anomaly detection models.

Therefore, StandardScaler was applied only to Time and Amount to ensure better model stability and fair distance-based learning.

In [ ]:
X = df_credit.drop("Class", axis=1)
y = df_credit["Class"]

print(X.shape)
print(y.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

X[["Time", "Amount"]] = scaler.fit_transform(
    X[["Time", "Amount"]]
)

X.head()